# SGLang 与 vLLM 推理速度、显存占用实测对比

**定位**：在同一硬件上，用 **相同模型、相同提示词** 分别对 [SGLang](https://github.com/sgl-project/sglang) 和 [vLLM](https://github.com/vllm-project/vllm) 进行基准测试，对比 **平均延迟、吞吐量、显存占用**，帮助你选择更适合的推理框架。

> **默认启动命令（SGLang）：**
> ```bash
> python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
> ```

*本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台。*

---

## 硬件与环境要求

| 项目 | 说明 |
|------|------|
| **GPU** | NVIDIA GPU（算力 ≥ 7.0），本 Demo 单卡即可 |
| **显存（VRAM）** | ≥ 6 GB（需足够加载一个 0.5B 模型实例；两个框架交替运行，非同时占用） |
| **驱动** | NVIDIA Driver ≥ 525，与 PyTorch/CUDA 版本匹配 |
| **CUDA** | ≥ 12.1（推荐 cu124） |
| **Python** | 3.9 – 3.12 |
| **操作系统** | Linux x86_64 最佳；Windows 建议使用 WSL2 |

---

## 依赖安装

```bash
# 0) 创建并激活虚拟环境
python3 -m venv .venv
source .venv/bin/activate

# 1) 升级 pip
python -m pip install --upgrade pip

# 2) 安装 PyTorch（CUDA 12.4 示例）
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 3) 同时安装 SGLang 和 vLLM
pip install "sglang[all]" "vllm>=0.6.0" "openai>=1.0.0" "requests>=2.28.0"

# 4) Jupyter
pip install notebook ipykernel
```

> **注意**：SGLang 和 vLLM 可能对 PyTorch 版本有不同要求，若安装冲突请参考各框架官方文档调整。

---

## 使用指南

1. 确保已安装 SGLang 和 vLLM 及其依赖。
2. 按顺序执行每个代码单元格：先定义工具函数，再依次测试 SGLang 和 vLLM。
3. 测试过程中 **不要同时运行两个框架**，以确保显存和计算资源不互相干扰。
4. 最后运行「对比结果汇总」查看表格，并执行「清理」释放资源。
5. 建议多次运行取平均值以获得更稳定的结果。

---

In [ ]:
# === 环境检查 ===

import sys  # 导入 sys 获取 Python 版本

print("Python 版本:", sys.version)  # 打印当前 Python 版本

try:
    import torch  # 尝试导入 PyTorch
except ImportError:
    print("未检测到 torch：请先安装带 CUDA 的 PyTorch。")  # 提示安装
else:
    print("torch 版本:", torch.__version__)  # 打印 torch 版本
    print("CUDA 是否可用:", torch.cuda.is_available())  # 检测 GPU 可用性
    if torch.cuda.is_available():  # 仅在有 GPU 时打印详细信息
        print("GPU0 名称:", torch.cuda.get_device_name(0))  # GPU 产品名
        vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)  # 计算总显存 GB
        print("GPU0 显存(GB):", vram_gb)  # 打印显存大小

In [ ]:
# 可选：在 Notebook 内核中安装依赖（若终端已装好可跳过）

import subprocess  # 导入 subprocess 用于执行 pip
import sys  # 导入 sys 获取当前解释器路径

_packages = ["sglang[all]", "vllm>=0.6.0", "openai>=1.0.0", "requests>=2.28.0"]  # 需要安装的包列表
_cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "pip"] + _packages  # 组装完整安装命令
subprocess.run(_cmd, check=True)  # 执行安装，失败时抛出异常
print("依赖安装完毕。若缺少 torch/cuda 请单独安装 PyTorch。")  # 提示完成

## 对比方法说明

为保证公平对比，两个框架使用 **完全相同** 的条件：

- **模型**：`Qwen/Qwen2.5-0.5B-Instruct`
- **提示词**：「请用100字介绍一下人工智能的发展历史。」
- **请求数量**：20 次顺序请求
- **测量指标**：平均延迟（秒/请求）、吞吐量（token/秒）、GPU 显存占用（MB）
- **测试流程**：启动框架 → 记录显存 → 运行基准 → 停止框架 → 测试下一个

---

## 工具函数：启动/停止服务 + 显存监控

---

In [ ]:
# 工具函数：启动/停止推理服务 + GPU 显存监控

import json  # 用于解析 HTTP JSON 响应
import os  # 用于进程组操作和信号发送
import signal  # 用于发送 SIGTERM 信号
import subprocess  # 用于创建和管理子进程
import sys  # 获取当前 Python 解释器路径
import time  # 用于计时和等待
import urllib.error  # 捕获 HTTP 异常
import urllib.request  # 发送 HTTP 请求探活

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # 两个框架使用同一个模型


def get_gpu_memory_mb():
    """通过 nvidia-smi 获取 GPU 0 当前已用显存（MB）。"""
    try:
        result = subprocess.run(  # 执行 nvidia-smi 查询命令
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
            capture_output=True, text=True, check=True,  # 捕获输出，文本模式，检查返回码
        )
        return int(result.stdout.strip())  # 解析并返回已用显存数值
    except Exception as e:  # nvidia-smi 不可用时回退
        print(f"nvidia-smi 查询失败: {e}")  # 打印错误信息
        return -1  # 返回 -1 表示无法获取


def launch_server(framework, port):
    """启动指定框架的推理服务并等待就绪。"""
    if framework == "sglang":  # SGLang 启动命令
        cmd = [  # 组装 SGLang 启动参数
            sys.executable, "-m", "sglang.launch_server",  # 以模块方式启动
            "--model-path", MODEL_ID,  # 指定模型
            "--host", "127.0.0.1",  # 绑定本机
            "--port", str(port),  # 指定端口
        ]
    elif framework == "vllm":  # vLLM 启动命令
        cmd = [  # 组装 vLLM 启动参数
            sys.executable, "-m", "vllm.entrypoints.openai.api_server",  # vLLM 的 OpenAI 兼容入口
            "--model", MODEL_ID,  # 指定模型
            "--host", "127.0.0.1",  # 绑定本机
            "--port", str(port),  # 指定端口
            "--dtype", "half",  # 使用 FP16 节省显存
        ]
    else:
        raise ValueError(f"不支持的框架: {framework}")  # 参数校验

    print(f"正在启动 {framework}（端口 {port}）...")  # 提示用户启动中
    proc = subprocess.Popen(  # 创建后台子进程
        cmd,
        stdout=subprocess.DEVNULL,  # 丢弃标准输出
        stderr=subprocess.STDOUT,  # 合并错误到标准输出
        start_new_session=True,  # 新建进程组
    )
    return proc  # 返回进程句柄


def wait_for_server(port, timeout=600):
    """轮询 /v1/models 端点直到服务就绪或超时。"""
    url = f"http://127.0.0.1:{port}/v1/models"  # 探活 URL
    deadline = time.time() + timeout  # 计算截止时间
    while time.time() < deadline:  # 在截止时间前持续轮询
        try:
            with urllib.request.urlopen(url, timeout=2) as resp:  # 发送 GET 请求
                json.loads(resp.read().decode("utf-8"))  # 尝试解析 JSON 确认有效
            print(f"端口 {port} 服务已就绪！")  # 服务可用
            return True  # 返回成功
        except Exception:  # 任何连接/解析错误都继续等待
            time.sleep(2)  # 等待 2 秒后重试
    raise TimeoutError(f"等待端口 {port} 超时（{timeout}s）")  # 超时报错


def stop_server(proc):
    """停止指定进程及其进程组。"""
    if proc is None or proc.poll() is not None:  # 进程不存在或已退出
        return  # 无需处理
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)  # 向整个进程组发送终止信号
    except ProcessLookupError:
        proc.terminate()  # 进程组操作失败则直接终止
    proc.wait(timeout=30)  # 等待最多 30 秒让进程退出
    print("服务已停止。")  # 告知用户


print("工具函数定义完毕。")  # 确认加载成功

## 基准测试函数

---

In [ ]:
# 基准测试函数：发送 N 次请求并统计延迟和吞吐量

from openai import OpenAI  # 导入 OpenAI SDK 客户端

BENCHMARK_PROMPT = "请用100字介绍一下人工智能的发展历史。"  # 统一测试用的提示词
NUM_REQUESTS = 20  # 总请求数量
MAX_TOKENS = 150  # 每次请求最大生成 token 数


def run_benchmark(port, num_requests=NUM_REQUESTS):
    """对指定端口的服务发送 num_requests 个请求，返回统计结果字典。"""
    client = OpenAI(  # 创建指向目标服务的客户端
        base_url=f"http://127.0.0.1:{port}/v1",  # 根据端口构建 base_url
        api_key="EMPTY",  # 本地服务无需真实密钥
    )

    latencies = []  # 存储每次请求的延迟时间（秒）
    total_tokens = 0  # 累计生成的 token 总数

    print(f"开始发送 {num_requests} 个请求...")  # 提示测试开始
    overall_start = time.time()  # 记录整体开始时间

    for i in range(num_requests):  # 逐个发送请求
        t0 = time.time()  # 记录单次请求开始时间
        completion = client.chat.completions.create(  # 发起聊天补全
            model=MODEL_ID,  # 指定模型
            messages=[  # 消息列表
                {"role": "system", "content": "你是一个知识丰富的中文助手。"},  # 系统角色设定
                {"role": "user", "content": BENCHMARK_PROMPT},  # 用户提示词
            ],
            temperature=0.3,  # 适中温度保证输出一致性
            max_tokens=MAX_TOKENS,  # 限制生成长度
        )
        t1 = time.time()  # 记录单次请求结束时间
        latencies.append(t1 - t0)  # 计算并存储延迟
        completion_tokens = completion.usage.completion_tokens  # 获取本次生成的 token 数
        total_tokens += completion_tokens  # 累加 token
        if (i + 1) % 5 == 0:  # 每 5 个请求打印一次进度
            print(f"  已完成 {i + 1}/{num_requests} 个请求")  # 进度提示

    overall_time = time.time() - overall_start  # 计算总耗时
    avg_latency = sum(latencies) / len(latencies)  # 计算平均延迟
    throughput = total_tokens / overall_time if overall_time > 0 else 0  # 计算吞吐量（token/s）

    result = {  # 组装结果字典
        "total_time": round(overall_time, 2),  # 总耗时（秒）
        "avg_latency": round(avg_latency, 3),  # 平均延迟（秒）
        "throughput": round(throughput, 1),  # 吞吐量（token/s）
        "total_tokens": total_tokens,  # 总生成 token 数
    }
    print(f"测试完成：总耗时 {result['total_time']}s，平均延迟 {result['avg_latency']}s，吞吐量 {result['throughput']} tok/s")  # 打印结果摘要
    return result  # 返回结果


print("基准测试函数定义完毕。")  # 确认加载成功

## 启动 SGLang 并测试

---

In [ ]:
# 启动 SGLang 服务，记录显存，运行基准测试，停止服务

import time  # 导入 time 用于等待显存稳定

SGLANG_PORT = 30000  # SGLang 使用的端口

mem_before_sglang = get_gpu_memory_mb()  # 记录启动前 GPU 显存
print(f"SGLang 启动前 GPU 显存: {mem_before_sglang} MB")  # 打印基线显存

sglang_proc = launch_server("sglang", SGLANG_PORT)  # 启动 SGLang 子进程
wait_for_server(SGLANG_PORT)  # 等待服务就绪

time.sleep(5)  # 等待 5 秒让显存分配稳定
mem_after_sglang = get_gpu_memory_mb()  # 记录启动后 GPU 显存
sglang_mem = mem_after_sglang - mem_before_sglang  # 计算 SGLang 占用的显存增量
print(f"SGLang 启动后 GPU 显存: {mem_after_sglang} MB（增量 {sglang_mem} MB）")  # 打印显存变化

print("\n--- SGLang 基准测试 ---")  # 测试开始提示
sglang_result = run_benchmark(SGLANG_PORT)  # 运行基准测试并保存结果
sglang_result["gpu_memory_mb"] = sglang_mem  # 将显存信息加入结果

stop_server(sglang_proc)  # 停止 SGLang 服务
time.sleep(5)  # 等待 GPU 显存完全释放
print("SGLang 测试完成并已停止服务。")  # 确认完成

## 启动 vLLM 并测试

---

In [ ]:
# 启动 vLLM 服务，记录显存，运行基准测试，停止服务

VLLM_PORT = 8000  # vLLM 使用的端口

mem_before_vllm = get_gpu_memory_mb()  # 记录启动前 GPU 显存
print(f"vLLM 启动前 GPU 显存: {mem_before_vllm} MB")  # 打印基线显存

vllm_proc = launch_server("vllm", VLLM_PORT)  # 启动 vLLM 子进程
wait_for_server(VLLM_PORT)  # 等待服务就绪

time.sleep(5)  # 等待 5 秒让显存分配稳定
mem_after_vllm = get_gpu_memory_mb()  # 记录启动后 GPU 显存
vllm_mem = mem_after_vllm - mem_before_vllm  # 计算 vLLM 占用的显存增量
print(f"vLLM 启动后 GPU 显存: {mem_after_vllm} MB（增量 {vllm_mem} MB）")  # 打印显存变化

print("\n--- vLLM 基准测试 ---")  # 测试开始提示
vllm_result = run_benchmark(VLLM_PORT)  # 运行基准测试并保存结果
vllm_result["gpu_memory_mb"] = vllm_mem  # 将显存信息加入结果

stop_server(vllm_proc)  # 停止 vLLM 服务
time.sleep(5)  # 等待 GPU 显存完全释放
print("vLLM 测试完成并已停止服务。")  # 确认完成

## 对比结果汇总

---

In [ ]:
# 汇总两个框架的测试结果并打印对比表格

print("=" * 70)  # 打印分隔线
print("SGLang vs vLLM 对比结果")  # 打印标题
print("=" * 70)  # 打印分隔线
print(f"模型: {MODEL_ID}")  # 显示测试用的模型
print(f"请求数: {NUM_REQUESTS}，每请求最大 token: {MAX_TOKENS}")  # 显示测试参数
print("-" * 70)  # 打印分隔线

header = f"{'指标':<20} {'SGLang':>15} {'vLLM':>15} {'差异':>15}"  # 格式化表头
print(header)  # 打印表头
print("-" * 70)  # 打印分隔线

rows = [  # 定义要展示的指标行
    ("总耗时(s)", sglang_result["total_time"], vllm_result["total_time"]),  # 总耗时对比
    ("平均延迟(s/req)", sglang_result["avg_latency"], vllm_result["avg_latency"]),  # 平均延迟对比
    ("吞吐量(tok/s)", sglang_result["throughput"], vllm_result["throughput"]),  # 吞吐量对比
    ("总生成 tokens", sglang_result["total_tokens"], vllm_result["total_tokens"]),  # 总 token 对比
    ("GPU 显存增量(MB)", sglang_result["gpu_memory_mb"], vllm_result["gpu_memory_mb"]),  # 显存对比
]

for name, sg_val, vl_val in rows:  # 遍历每行指标
    if isinstance(sg_val, (int, float)) and isinstance(vl_val, (int, float)) and vl_val != 0:  # 确保数值有效
        if name == "吞吐量(tok/s)":  # 吞吐量越高越好
            ratio = f"{sg_val / vl_val:.2f}x"  # 计算 SGLang 相对 vLLM 的倍数
        else:  # 延迟和显存越低越好
            ratio = f"{sg_val / vl_val:.2f}x"  # 计算比值
    else:
        ratio = "N/A"  # 无法计算时显示 N/A
    print(f"{name:<20} {str(sg_val):>15} {str(vl_val):>15} {ratio:>15}")  # 格式化输出一行

print("=" * 70)  # 打印结束分隔线
print("\n注意：以上结果受硬件、驱动版本、模型大小等因素影响，仅供参考。")  # 免责声明

## 清理资源

---

In [ ]:
# 确保所有服务进程已停止，释放 GPU 显存

import os  # 导入 os 用于进程组操作
import signal  # 导入 signal 用于发送终止信号

for name, proc in [("SGLang", sglang_proc), ("vLLM", vllm_proc)]:  # 遍历两个框架的进程
    if proc is not None and proc.poll() is None:  # 检查进程是否仍在运行
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)  # 向进程组发送终止信号
        except ProcessLookupError:
            proc.terminate()  # 进程组操作失败则直接终止
        proc.wait(timeout=30)  # 等待进程退出
        print(f"{name} 进程已停止。")  # 确认停止
    else:
        print(f"{name} 进程已不在运行。")  # 进程已退出

print("所有资源已清理完毕。")  # 最终确认

## 常见报错与排查

### 1）某个框架启动失败 / 超时无法就绪

**原因**：SGLang 和 vLLM 对 PyTorch / CUDA 版本可能有不同要求，某些版本组合会导致其中一个框架无法正常启动。

**解决**：
- 检查两个框架各自的安装文档，确认 Python、PyTorch、CUDA 版本兼容
- 在终端手动运行启动命令查看完整错误日志
- 确保 GPU 驱动足够新（建议 Driver ≥ 525）

### 2）测试结果波动很大 / 不稳定

**原因**：GPU 上有其他进程占用计算资源或显存，导致测试结果受干扰。

**解决**：
- 运行 `nvidia-smi` 确认没有其他 GPU 进程
- 关闭浏览器等可能使用 GPU 的应用
- 多运行几次取平均值
- 确保两个框架的测试之间有足够的间隔让 GPU 冷却

---

**结语**：本 Notebook 提供了 SGLang 与 vLLM 的基础对比方法。实际生产环境中，还需考虑并发性能、长文本处理、不同模型规模等更多维度的评估。